In [1]:
import os
import ast
import torch
import torchaudio
import torchaudio.transforms as T
import pandas as pd
import numpy as np
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit

BASE = "/kaggle/input/competitions/birdclef-2026"
TRAIN_AUDIO_DIR = os.path.join(BASE, "train_audio")
TRAIN_CSV = os.path.join(BASE, "train.csv")
TAXONOMY_CSV = os.path.join(BASE, "taxonomy.csv")
SOUNDSCAPE_DIR = os.path.join(BASE, "train_soundscapes")
SOUNDSCAPE_CSV = os.path.join(BASE, "train_soundscapes_labels.csv")
WORK_DIR = "/kaggle/working"
SMART_CROP_CSV = os.path.join(WORK_DIR, "smart_crop_ranges.csv")

os.makedirs(WORK_DIR, exist_ok=True)

class Config:
    SR = 32000               
    DURATION = 5             
    MAX_LENGTH = SR * DURATION 
    N_MELS = 128             
    N_FFT = 1024
    HOP_LENGTH = 512
    BATCH_SIZE = 32
    NUM_WORKERS = 4
    SMART_CROP_HOP_SEC = 0.5
    

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
CLASSES = taxonomy_df["primary_label"].unique().tolist()
NUM_CLASSES = len(CLASSES)
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

train_df = pd.read_csv(TRAIN_CSV)
soundscape_df = pd.read_csv(SOUNDSCAPE_CSV)


def to_seconds(ts):
    if isinstance(ts, (int, float)):
        return float(ts)

    parts = str(ts).split(":")

    if len(parts) == 3:
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])

    if len(parts) == 2:
        return int(parts[0]) * 60 + float(parts[1])

    return float(ts)


train_rows = []

for _, row in train_df.iterrows():
    labels = [row["primary_label"]]

    sec_labels = ast.literal_eval(row.get("secondary_labels", "[]"))
    labels.extend(sec_labels)

    train_rows.append({
        "filename": row["filename"],
        "audio_path": os.path.join(TRAIN_AUDIO_DIR, row["filename"]),
        "all_labels": list(set(labels)),
        "is_soundscape": 0,
        "start_sec": 0.0,
    })


soundscape_rows = []

for _, row in soundscape_df.iterrows():
    labels = [
        lab for lab in str(row["primary_label"]).split(";")
        if lab in class_to_idx
    ]

    soundscape_rows.append({
        "filename": row["filename"],
        "audio_path": os.path.join(SOUNDSCAPE_DIR, row["filename"]),
        "all_labels": labels,
        "is_soundscape": 1,
        "start_sec": to_seconds(row["start"]),
    })


clean_manifest = pd.DataFrame(train_rows)
soundscape_manifest = pd.DataFrame(soundscape_rows)


def find_best_5sec_range_for_file(audio_path, config):
    waveform, sr = torchaudio.load(audio_path)

    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    audio_len = waveform.shape[1]
    window_size = int(config.DURATION * sr)
    duration_sec = audio_len / sr

    if audio_len <= window_size:
        smart_start = 0
        smart_end = audio_len
        smart_energy = waveform.pow(2).mean().item()
    else:
        hop_size = max(1, int(config.SMART_CROP_HOP_SEC * sr))
        max_start = audio_len - window_size

        starts = torch.arange(0, max_start + 1, hop_size)

        if starts[-1].item() != max_start:
            starts = torch.cat([starts, torch.tensor([max_start])])

        energy = waveform.pow(2).mean(dim=0)

        cumulative_energy = torch.cat([
            torch.zeros(1),
            energy.cumsum(dim=0),
        ])

        window_scores = cumulative_energy[starts + window_size] - cumulative_energy[starts]

        best_idx = torch.argmax(window_scores)

        smart_start = int(starts[best_idx].item())
        smart_end = smart_start + window_size
        smart_energy = (window_scores[best_idx] / window_size).item()

    return {
        "source_sr": sr,
        "duration_sec": duration_sec,
        "smart_start_sec": smart_start / sr,
        "smart_end_sec": smart_end / sr,
        "smart_energy": smart_energy,
    }


smart_crop_columns = [
    "source_sr",
    "duration_sec",
    "smart_start_sec",
    "smart_end_sec",
    "smart_energy",
    "filename",
]

if os.path.exists(SMART_CROP_CSV):
    smart_crop_df = pd.read_csv(SMART_CROP_CSV)
else:
    smart_crop_df = pd.DataFrame(columns=smart_crop_columns)

smart_crop_df = smart_crop_df.drop_duplicates(subset=["filename"], keep="last")
done_files = set(smart_crop_df["filename"].dropna().tolist())

remaining_manifest = clean_manifest[
    ~clean_manifest["filename"].isin(done_files)
].reset_index(drop=True)

smart_crop_rows = []
SAVE_EVERY = 300

for row in tqdm(
    remaining_manifest.itertuples(index=False),
    total=len(remaining_manifest),
    desc="Precomputing smart crops",
    leave=True,
):
    try:
        crop_info = find_best_5sec_range_for_file(row.audio_path, Config)
        crop_info["filename"] = row.filename
        smart_crop_rows.append(crop_info)

    except Exception:
        smart_crop_rows.append({
            "source_sr": Config.SR,
            "duration_sec": np.nan,
            "smart_start_sec": 0.0,
            "smart_end_sec": Config.DURATION,
            "smart_energy": np.nan,
            "filename": row.filename,
        })

    if len(smart_crop_rows) >= SAVE_EVERY:
        smart_crop_df = pd.concat(
            [smart_crop_df, pd.DataFrame(smart_crop_rows)],
            ignore_index=True,
        )

        smart_crop_df = smart_crop_df.drop_duplicates(
            subset=["filename"],
            keep="last",
        )

        smart_crop_df.to_csv(SMART_CROP_CSV, index=False)
        smart_crop_rows = []


if len(smart_crop_rows) > 0:
    smart_crop_df = pd.concat(
        [smart_crop_df, pd.DataFrame(smart_crop_rows)],
        ignore_index=True,
    )

    smart_crop_df = smart_crop_df.drop_duplicates(
        subset=["filename"],
        keep="last",
    )

    smart_crop_df.to_csv(SMART_CROP_CSV, index=False)


clean_manifest = clean_manifest.merge(
    smart_crop_df,
    on="filename",
    how="left",
)

missing_smart_ranges = clean_manifest["smart_start_sec"].isna().sum()

assert missing_smart_ranges == 0, (
    f"Missing smart ranges for {missing_smart_ranges} files. "
    "Rerun this cell before training."
)


soundscape_manifest["source_sr"] = Config.SR
soundscape_manifest["duration_sec"] = np.nan
soundscape_manifest["smart_start_sec"] = soundscape_manifest["start_sec"]
soundscape_manifest["smart_end_sec"] = soundscape_manifest["start_sec"] + Config.DURATION
soundscape_manifest["smart_energy"] = np.nan

# Oversampling: add soundscapes twice, as in the original pipeline
train_manifest = pd.concat(
    [clean_manifest, soundscape_manifest, soundscape_manifest],
    ignore_index=True,
)


def rebuild_audio_path(row):
    if int(row["is_soundscape"]) == 1:
        return os.path.join(SOUNDSCAPE_DIR, row["filename"])

    return os.path.join(TRAIN_AUDIO_DIR, row["filename"])


train_manifest["audio_path"] = train_manifest.apply(rebuild_audio_path, axis=1)

assert train_manifest["filename"].isna().sum() == 0
assert train_manifest["audio_path"].isna().sum() == 0


gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42,
)

train_idx, val_idx = next(
    gss.split(
        train_manifest,
        groups=train_manifest["filename"],
    )
)

train_split = train_manifest.iloc[train_idx].reset_index(drop=True)
val_split = train_manifest.iloc[val_idx].reset_index(drop=True)

Precomputing smart crops:   1%|          | 299/35549 [00:14<17:42, 33.17it/s] /tmp/ipykernel_57/332083103.py:190: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  smart_crop_df = pd.concat(
Precomputing smart crops: 100%|██████████| 35549/35549 [23:42<00:00, 25.00it/s]  


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU 0:", torch.cuda.get_device_name(0))

CUDA available: True
Device count: 2
GPU 0: Tesla T4


In [3]:
print("clean_manifest rows:", len(clean_manifest))
print("SMART_CROP_CSV exists:", os.path.exists(SMART_CROP_CSV))

if os.path.exists(SMART_CROP_CSV):
    tmp = pd.read_csv(SMART_CROP_CSV)
    print("smart_crop_df rows:", len(tmp))
    print("unique filenames in smart_crop_df:", tmp["filename"].nunique())
    print("missing files:", len(clean_manifest[~clean_manifest["filename"].isin(tmp["filename"])]))
    display(tmp.head())
else:
    print("No smart crop CSV yet")

clean_manifest rows: 35549
SMART_CROP_CSV exists: True
smart_crop_df rows: 35549
unique filenames in smart_crop_df: 35549
missing files: 0


,source_sr,duration_sec,smart_start_sec,smart_end_sec,smart_energy,filename
0,32000,18.026,4.5,9.5,0.000342,1161364/iNat1216197.ogg
1,32000,28.398,2.5,7.5,0.000154,1161364/iNat1114648.ogg
2,32000,80.016,5.0,10.0,0.000228,1161364/iNat810195.ogg
3,32000,61.272,0.5,5.5,0.010661,1161364/iNat818781.ogg
4,32000,71.064,54.5,59.5,0.001933,1161364/iNat556514.ogg


In [4]:
class BirdCLEFDataset(Dataset):
    def __init__(self, df, config, is_train=True):
        self.df = df
        self.config = config
        self.is_train = is_train
        
        self.mel_transform = T.MelSpectrogram(
            sample_rate=config.SR,
            n_fft=config.N_FFT,
            hop_length=config.HOP_LENGTH,
            n_mels=config.N_MELS,
            f_min=50,
            f_max=14000
        )
        self.amp_to_db = T.AmplitudeToDB()
        
        if self.is_train:
            self.time_masking = T.TimeMasking(time_mask_param=30)
            self.freq_masking = T.FrequencyMasking(freq_mask_param=20)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = row['audio_path']
        
        if row['is_soundscape'] == 1:
            start_frame = int(row['start_sec'] * self.config.SR)
            num_frames = self.config.MAX_LENGTH
            waveform, sr = torchaudio.load(audio_path, frame_offset=start_frame, num_frames=num_frames)
        else:
            source_sr = int(row['source_sr'])
            start_frame = int(row['smart_start_sec'] * source_sr)
            num_frames = int(self.config.DURATION * source_sr)
            waveform, sr = torchaudio.load(audio_path, frame_offset=start_frame, num_frames=num_frames)
            
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        audio_len = waveform.shape[1]
        
        if audio_len > self.config.MAX_LENGTH:
            waveform = waveform[:, :self.config.MAX_LENGTH]
            
        elif audio_len < self.config.MAX_LENGTH:
            pad_len = self.config.MAX_LENGTH - audio_len
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))
            
        mel_spec = self.mel_transform(waveform)
        mel_spec = self.amp_to_db(mel_spec)
        
        # Augmentation
        if self.is_train:
            mel_spec = self.time_masking(mel_spec)
            mel_spec = self.freq_masking(mel_spec)
            
            noise = torch.randn_like(mel_spec) * 0.1 * (mel_spec.max() - mel_spec.min())
            mel_spec = mel_spec + noise
        
        mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min() + 1e-6)
        mel_spec = mel_spec * 2 - 1
        mel_spec = mel_spec.expand(3, -1, -1)
        
        target = torch.zeros(NUM_CLASSES, dtype=torch.float32)
        for label in row['all_labels']:
            if label in class_to_idx:
                target[class_to_idx[label]] = 1.0

        return mel_spec, target

In [5]:
train_dataset = BirdCLEFDataset(train_split, Config, is_train=True)
val_dataset = BirdCLEFDataset(val_split, Config, is_train=False)

train_loader = DataLoader(
    train_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=True, 
    num_workers=Config.NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=False, 
    num_workers=Config.NUM_WORKERS,
    pin_memory=True
)

In [6]:
import torch.nn as nn
import timm

class BirdCLEFModel(nn.Module):
    def __init__(self, model_name='efficientnet_b0', num_classes=NUM_CLASSES, pretrained=True):
        super().__init__()
        
        self.backbone = timm.create_model(
            model_name, 
            pretrained=pretrained, 
            num_classes=0 
        )
        
        in_features = self.backbone.num_features
        
        self.head = nn.Linear(in_features, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        
        logits = self.head(features)
        
        return logits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BirdCLEFModel().to(device)
criterion = nn.BCEWithLogitsLoss()

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [7]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
Device count: 2
GPU: Tesla T4


In [8]:
import time
import numpy as np
import torch
import pandas as pd
from sklearn.metrics import roc_auc_score
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

def calculate_competition_roc_auc(y_true, y_pred):
    aucs = []
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) == 2:
            class_auc = roc_auc_score(y_true[:, i], y_pred[:, i])
            aucs.append(class_auc)
    if len(aucs) == 0:
        return 0.5
    return np.mean(aucs)

EPOCHS = 10
best_val_auc = 0.0

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

history = []

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    
    for images, targets in train_pbar:
        images = images.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()
        
        logits = model(images)
        loss = criterion(logits, targets)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        train_pbar.set_postfix(batch_loss=f"{loss.item():.4f}")
        
    train_loss = train_loss / len(train_loader.dataset)
    
    model.eval()
    val_loss = 0.0
    all_val_targets = []
    all_val_preds = []
    
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False)
    
    with torch.no_grad():
        for images, targets in val_pbar:
            images = images.to(device)
            targets = targets.to(device)
            
            logits = model(images)
            loss = criterion(logits, targets)
            val_loss += loss.item() * images.size(0)
            
            probs = torch.sigmoid(logits)
            all_val_targets.append(targets.cpu().numpy())
            all_val_preds.append(probs.cpu().numpy())
            
            val_pbar.set_postfix(batch_loss=f"{loss.item():.4f}")
            
    val_loss = val_loss / len(val_loader.dataset)
    all_val_targets = np.vstack(all_val_targets)
    all_val_preds = np.vstack(all_val_preds)
    
    val_auc = calculate_competition_roc_auc(all_val_targets, all_val_preds)
    
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()
    
    print(f"Epoch {epoch+1}/{EPOCHS} | LR: {current_lr:.2e}")
    print(f"  -> Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val ROC-AUC: {val_auc:.4f}")
    
    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_auc': val_auc,
        'lr': current_lr
    })
    
    pd.DataFrame(history).to_csv(f"{WORK_DIR}training_history.csv", index=False)
    
    if val_auc > best_val_auc:
        print(f"  [+] Validation AUC improved ({best_val_auc:.4f} -> {val_auc:.4f}). Saving best model!")
        best_val_auc = val_auc
        torch.save(model.state_dict(), f"{WORK_DIR}best_birdclef_model.pth")

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'val_auc': val_auc
    }, f"{WORK_DIR}last_checkpoint.pth")

Epoch 1/10 | LR: 1.00e-03
  -> Train Loss: 0.0318 | Val Loss: 0.0388 | Val ROC-AUC: 0.7465
  [+] Validation AUC improved (0.0000 -> 0.7465). Saving best model!


Epoch 2/10 | LR: 9.76e-04
  -> Train Loss: 0.0209 | Val Loss: 0.0218 | Val ROC-AUC: 0.8975
  [+] Validation AUC improved (0.7465 -> 0.8975). Saving best model!


Epoch 3/10 | LR: 9.05e-04
  -> Train Loss: 0.0177 | Val Loss: 0.0198 | Val ROC-AUC: 0.9176
  [+] Validation AUC improved (0.8975 -> 0.9176). Saving best model!


Epoch 4/10 | LR: 7.94e-04
  -> Train Loss: 0.0156 | Val Loss: 0.0203 | Val ROC-AUC: 0.9102


Epoch 5/10 | LR: 6.55e-04
  -> Train Loss: 0.0140 | Val Loss: 0.0183 | Val ROC-AUC: 0.9209
  [+] Validation AUC improved (0.9176 -> 0.9209). Saving best model!


Epoch 6/10 | LR: 5.01e-04
  -> Train Loss: 0.0126 | Val Loss: 0.0185 | Val ROC-AUC: 0.9282
  [+] Validation AUC improved (0.9209 -> 0.9282). Saving best model!


Epoch 7/10 | LR: 3.46e-04
  -> Train Loss: 0.0112 | Val Loss: 0.0184 | Val ROC-AUC: 0.9246


Epoch 8/10 | LR: 2.07e-04
  -> Train Loss: 0.0099 | Val Loss: 0.0182 | Val ROC-AUC: 0.9300
  [+] Validation AUC improved (0.9282 -> 0.9300). Saving best model!


Epoch 9/10 | LR: 9.64e-05
  -> Train Loss: 0.0089 | Val Loss: 0.0180 | Val ROC-AUC: 0.9335
  [+] Validation AUC improved (0.9300 -> 0.9335). Saving best model!


Epoch 10/10 | LR: 2.54e-05
  -> Train Loss: 0.0083 | Val Loss: 0.0183 | Val ROC-AUC: 0.9317


In [10]:
from pathlib import Path

for path in [
    "/kaggle/working/best_birdclef_model.pth",
    "/kaggle/workingbest_birdclef_model.pth",
    "/kaggle/working/last_checkpoint.pth",
    "/kaggle/workinglast_checkpoint.pth",
]:
    p = Path(path)
    print(path, "exists:", p.exists(), "size MB:", round(p.stat().st_size / 1024 / 1024, 2) if p.exists() else None)

/kaggle/working/best_birdclef_model.pth exists: False size MB: None
/kaggle/workingbest_birdclef_model.pth exists: True size MB: 16.72
/kaggle/working/last_checkpoint.pth exists: False size MB: None
/kaggle/workinglast_checkpoint.pth exists: True size MB: 49.77


In [11]:
import os
import shutil
from pathlib import Path

BROKEN_MODEL_PATH = "/kaggle/workingbest_birdclef_model.pth"
FIXED_MODEL_PATH = "/kaggle/working/best_birdclef_model.pth"

assert os.path.exists(BROKEN_MODEL_PATH), "Model not found."

shutil.copy(BROKEN_MODEL_PATH, FIXED_MODEL_PATH)

print("Copied model to:", FIXED_MODEL_PATH)
print("Exists:", os.path.exists(FIXED_MODEL_PATH))
print("Size MB:", round(os.path.getsize(FIXED_MODEL_PATH) / 1024 / 1024, 2))

Copied model to: /kaggle/working/best_birdclef_model.pth
Exists: True
Size MB: 16.72


In [ ]:
import os
WORK_DIR = "/kaggle/working"
MODEL_PATH = f"{WORK_DIR}best_birdclef_model.pth"

print(MODEL_PATH)
print(os.path.exists(MODEL_PATH))
print(os.path.getsize(MODEL_PATH) / 1024 / 1024, "MB")

In [ ]:
model.load_state_dict(torch.load(f'{WORK_DIR}best_birdclef_model.pth', map_location=device))
model.to(device)
model.eval()

In [ ]:
import glob
import os
import torch
import torchaudio
import pandas as pd

TEST_AUDIO_DIR = os.path.join(BASE, "test_soundscapes")
test_files = glob.glob(os.path.join(TEST_AUDIO_DIR, "*.ogg"))

if len(test_files) == 0:
    TEST_AUDIO_DIR = os.path.join(BASE, "train_soundscapes")
    test_files = glob.glob(os.path.join(TEST_AUDIO_DIR, "*.ogg"))[:2] 

model.eval()
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=Config.SR, 
    n_fft=Config.N_FFT, 
    hop_length=Config.HOP_LENGTH, 
    n_mels=Config.N_MELS, 
    f_min=50, 
    f_max=14000
).to(device)
amp_to_db = torchaudio.transforms.AmplitudeToDB().to(device)

predictions = []

with torch.no_grad():
    for file_path in test_files:
        filename = os.path.basename(file_path)
        file_id = filename.replace('.ogg', '')
        
        waveform, sr = torchaudio.load(file_path)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        waveform = waveform.to(device)
        
        chunk_length = Config.MAX_LENGTH # 160,000 samples
        num_chunks = waveform.shape[1] // chunk_length
        
        if num_chunks == 0:
            pad_len = chunk_length - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))
            num_chunks = 1
            
        for i in range(num_chunks):
            start = i * chunk_length
            end = start + chunk_length
            chunk = waveform[:, start:end]
            
            mel_spec = mel_transform(chunk)
            mel_spec = amp_to_db(mel_spec)
            mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min() + 1e-6)
            mel_spec = mel_spec * 2 - 1
            
            mel_spec = mel_spec.unsqueeze(0).expand(-1, 3, -1, -1)
            
            logits = model(mel_spec)
            probs = torch.sigmoid(logits).cpu().numpy()[0]
            
            end_time = (i + 1) * 5 # e.g., 5, 10, 15... 60
            row_id = f"{file_id}_{end_time}"
            
            pred_dict = {'row_id': row_id}
            for class_name, prob in zip(CLASSES, probs):
                pred_dict[class_name] = prob
                
            predictions.append(pred_dict)

submission_df = pd.DataFrame(predictions)
submission_df.to_csv(f'{WORK_DIR}submission.csv', index=False)
print(f'{WORK_DIR}submission.csv')